In [ ]:
!wget "https://surfdrive.surf.nl/files/index.php/s/G8XWW8On5MWhtCk/download?path=%2FInference&files=inference_results_ToyExample-v2_Cologne-v1_CologneBonnDusseldorf-v1.csv" -O "data/inference_results_ToyExample-v2_Cologne-v1_CologneBonnDusseldorf-v1.csv"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

In [ ]:
# Define baseline heuristic values (absolute values) for each environment
baseline_heuristic_values = {
    "ToyExample-v2": -276.552551 * 1e6,
    "Cologne-v1": -8192.396484 * 1e6,
    "CologneBonnDusseldorf-v1": -33102.484375 * 1e6,
}

def dataframe_to_best_checkpoints(data, env_name):
    """
    This function takes a dataframe and returns the best checkpoints for each algorithm.
    """
    # Get unique algorithms
    algorithms = data['algorithm'].unique()

    runs_per_algorithm = {
        alg: data[data['algorithm'] == alg]['WANDB_RUN_ID'].unique()
        for alg in algorithms
    }

    best_checkpoint_per_algorithm = {
        alg: {
            run: data[data['WANDB_RUN_ID'] == run]['mean'].max()
            for run in runs_per_algorithm[alg]
        }
    
        for alg in algorithms 
    }

    # Normalize the rewards by baseline heuristic results:
    heuristic_score = baseline_heuristic_values[env_name]

    best_checkpoint_per_algorithm_norm = {
        alg: {
            run: (score - heuristic_score) / abs(heuristic_score)
            for run, score in best_checkpoint_per_algorithm[alg].items()
        }
        for alg in algorithms
    }
    return best_checkpoint_per_algorithm_norm

data = pd.read_csv('data/inference_results_ToyExample-v2_Cologne-v1_CologneBonnDusseldorf-v1.csv')

algorithms = data['algorithm'].unique()
environments = data['map_name'].unique()

best_checkpoints = {
    environment: dataframe_to_best_checkpoints(data[data['map_name'] == environment], environment)
    for environment in environments
}

In [ ]:
from time import time
np.random.seed(int(time()))
random_seed = 45081 # np.random.randint(0, 100000)
#print(f"Random seed: {random_seed}")
figsize = (5.5, 3.8)
y_lims = (-55, 30)
x_margin = 0.75

print_extra_baseline_heuristics = False

algorithm_names = {
    "vdn_rnn": "VDN",
    "qmix_rnn": "QMIX",
    "pqn_rnn": "PQN-VDN",
    "ippo_rnn": "IPPO",
    "mappo_rnn": "MAPPO",
}

env_plot_titles = {
    "ToyExample-v2": "ToyExample\n(12 agents)",
    "Cologne-v1": "Cologne\n(60 agents)",
    "CologneBonnDusseldorf-v1": "CologneBonnDusseldorf\n(178 agents)",
}

plt.rcParams.update({
    'font.size': 8,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],  # or 'Times'
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'text.usetex': False,
})

In [ ]:
best_checkpoint_per_algorithm_norms = list(best_checkpoints.values())
np.random.seed(random_seed)
plot_titles = [env_plot_titles[env] for env in best_checkpoints.keys()] # Replace with actual titles
env_names = list(best_checkpoints.keys())  # Get the environment names for baseline values

# Create a figure with 3 subplots side by side
fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

# Loop through each subplot and dataset
for i, (ax, norm_data, title, env_name) in enumerate(zip(axes, best_checkpoint_per_algorithm_norms, plot_titles, env_names)):
    # Create DataFrame for this plot
    plot_data_norm = []
    for alg, runs in norm_data.items():
        for run, value in runs.items():
            if pd.notna(value) and np.isfinite(value):
                plot_data_norm.append({
                    'Algorithm': alg,
                    'Performance Improvement (%)': value * 100  # Convert to percentage
                })

    plot_df_norm = pd.DataFrame(plot_data_norm)
    
    dict_props = dict(linewidth=1)

    # Create boxplot on the current axis
    sns.boxplot(
        x='Algorithm', 
        y='Performance Improvement (%)', 
        data=plot_df_norm,
        palette='viridis', 
        ax=ax,
        showfliers=False,
        boxprops=dict_props,
        medianprops=dict_props,
        whiskerprops=dict_props,
        capprops=dict_props,
        #width=0.5,
    )

    ax.grid(True, axis="both", alpha=0.3)
    ax.set_axisbelow(True)

    # Map x-tick labels to friendly algorithm names
    ax.set_xticks(range(len(plot_df_norm['Algorithm'].unique())))
    ax.set_xticklabels([algorithm_names[alg] for alg in plot_df_norm['Algorithm'].unique()])
    
    # Set y-limits and get actual limits
    ax.set_ylim(y_lims)
    y_min, y_max = ax.get_ylim()
    
    # Dictionary to collect out-of-range points for each algorithm
    out_of_range_points = {alg: [] for alg in plot_df_norm['Algorithm'].unique()}
    
    def custom_y_labels(y, pos):
        if y == 0:
            return '0%'
        elif y == y_min:
            return f"≤ {y:.0f}%"
        else:
            return f"{y:+.1f}%"
        

    ax.yaxis.set_major_formatter(mticker.FuncFormatter(custom_y_labels))

    # Add data points
    for j, alg in enumerate(plot_df_norm['Algorithm'].unique()):
        alg_data = plot_df_norm[plot_df_norm['Algorithm'] == alg]['Performance Improvement (%)']
        x_jittered = np.zeros(len(alg_data))
        in_range_mask = np.ones(len(alg_data), dtype=bool)

        y_vals = alg_data.values
        sorted_indices = np.argsort(y_vals)
        y_vals = y_vals[sorted_indices]
        for k, y_val in enumerate(y_vals):
            indices = list(range(len(y_vals)))
            indices.remove(k)
            dist = abs(y_val - y_vals[indices])
            close_points = (dist < 1).sum()

            x_jittered[k] = j
            if close_points > 0:
                x_jittered[k] = j + np.sin(3 * np.pi * (y_val + k*np.sqrt(2) + np.random.normal(0, 1))) * 0.3 # Jitter the x position slightly
            #if close_points > 0:
            #    x_jittered[k] += np.random.uniform(0, 0.1)  # Add random jitter to x position
            #x_jittered[k] = np.clip(x_jittered[k], j - 0.3, j + 0.3)  # Ensure jittered values stay within bounds
            
            # Check if point is out of y-range (below y_min)
            if y_val < y_min:
                out_of_range_points[alg].append((x_jittered[k], y_val))
                in_range_mask[k] = False

        # Plot in-range points
        ax.scatter(
            x_jittered[in_range_mask], 
            y_vals[in_range_mask], 
            color='black',
            edgecolors='none',
            alpha=0.6, 
            s=10
        )
        
        # Add best and worst performance labels
        if len(alg_data) > 0:
            best_val = alg_data.max()
            worst_val = alg_data.min()
            
            # Format values to 1 decimal place
            best_text = f"{best_val:+.1f}%"
            worst_text = f"{worst_val:+.1f}%"
            
            # For the best value
            if best_val <= y_max:  # If the best value is within the visible range
                # Use annotate for normal cases (in view)
                ax.annotate(best_text, 
                            xy=(j, best_val), 
                            xytext=(0, 5), 
                            textcoords='offset points', 
                            ha='center', 
                            va='bottom',
                            fontsize=6, 
                            fontweight='bold',
                            color='green')
            else:  # If the best value is outside the visible range
                # Use text at the top of the visible area
                best_text = f"↑\n{best_text}"
                ax.text(j, y_max - 5, best_text, 
                        ha='center', 
                        va='center',
                        fontsize=6, 
                        fontweight='bold',
                        color='green')
            
            # For the worst value
            if worst_val >= y_min:  # If the worst value is within the visible range
                # Use annotate for normal cases (in view)
                ax.annotate(worst_text, 
                            xy=(j, worst_val), 
                            xytext=(0, -5), 
                            textcoords='offset points', 
                            ha='center',
                            va='top', 
                            fontsize=6, 
                            fontweight='bold',
                            color='red')
            else:  # If the worst value is outside the visible range
                # Use text at the bottom of the visible area
                worst_text = f"{worst_text}\n↓"
                ax.text(j, y_min + 5, worst_text, 
                        ha='center', 
                        va='center',
                        fontsize=6, 
                        fontweight='bold',
                        color='red')
    
    # Add baselines for each plot
    #base_lines_percentages = {k: v * 100 for k, v in base_lines_normalized.items()}
    
    # Add horizontal line at 0 (Humble Heuristic baseline)
    ax.axhline(
        y=0,
        color='red',
        linestyle='-',
        linewidth=1,
        label="Prioritized Heuristic $\\text{H}_\\text{PS}$",
        #zorder=-10
    )
    
    # Add the baseline value in millions on top of the red line
    baseline_value = baseline_heuristic_values[env_name]
    # Format the baseline value for display
    if abs(baseline_value) >= 1e9:
        # Convert to billions if value is larger than 1 billion
        baseline_text = f"$\\text{{H}}_\\text{{PS}}$ = {baseline_value/1e9:.1f}B"
    else:
        # Convert to millions
        baseline_text = f"$\\text{{H}}_\\text{{PS}}$ = {baseline_value/1e6:.1f}M"

    # Position the baseline text above the red line
    baseline_x_pos = -0.6 #len(plot_df_norm['Algorithm'].unique()) / 2 - 0.5  # Center of the plot
    ax.annotate(baseline_text,
                xy=(baseline_x_pos, -1),
                xytext=(0, 0),
                textcoords='offset points',
                ha='left',
                va='top',
                color='red',
                )
    

    if i == 0:
        # Add special bottom y-tick for out-of-range values
        
        # Get current y-ticks and add our custom one at the bottom
        yticks = list(ax.get_yticks())
        yticks.append(y_min)
        
        ax.set_yticks(yticks)
    
    # Plot all out-of-range points at the special tick position
    for alg, points in out_of_range_points.items():
        j = list(plot_df_norm['Algorithm'].unique()).index(alg)
        for x_pos, original_val in points:
            # Plot point at the special tick position
            ax.scatter(x_pos, y_min, color='red', alpha=0.7, s=15, marker='v', edgecolors='none')
    
    # Set titles and labels
    ax.set_title(title, fontsize=9) # 9
    ax.set_xlabel('', fontsize=9) 
    if i == 0:  # Only add y-label to the leftmost plot
        ax.set_ylabel('Relative Performance (%)', fontsize=8)
    else:
        ax.set_ylabel('')
    
    
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlim(-x_margin, len(plot_df_norm['Algorithm'].unique()) -1 + x_margin)

    if i == 0:
        ax.legend(loc='lower left', fontsize=7)

plt.tight_layout()
plt.subplots_adjust(wspace=0.0) # Adjust space between subplots

# Display the plot
plt.savefig('figures/Figure_3.pdf', bbox_inches='tight')
plt.savefig('figures/Figure_3.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
order = ['pqn_rnn', 'vdn_rnn', 'qmix_rnn', 'mappo_rnn', 'ippo_rnn']
for env in best_checkpoints.keys():
    print(f"Best checkpoints for {env}:")
    # Sort the algorithms by our custom order
    sorted_algorithms = sorted(best_checkpoints[env].keys(), key=lambda x: order.index(x))
    for alg in sorted_algorithms:
        runs = best_checkpoints[env][alg]
        # Sort runs by their values
        sorted_runs = sorted(runs.items(), key=lambda x: x[1], reverse=True)
        # Print the sorted runs
        # compute IQR
        values = np.array(list(runs.values()))
        q75, q25 = np.percentile(values, [75, 25])
        iqr = q75 - q25
        print(f"  {algorithm_names[alg]} & {' & '.join(f'{value*100:+.2f}' for run, value in sorted_runs)}")
        print(f"  IQR: {iqr*100:.2f}")
    print()